# 24 · PyTorch en pratique 2 : améliorer, chercher, CNN et RNN — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch24_pytorch_2/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch24_pytorch_2/03_notebook.ipynb`, créée par `python tools/start_chapter.py 24`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 24.Q1 à 24.Q12, 24.R1 à 24.R3, 24.1 à 24.5, 24.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 24.11 à 24.15 | améliorer un MLP sur Fashion-MNIST : taille de batch, quatre MLP, dropout et max-norm, validation au-dessus du train, journal d'expériences (*prochaine session de génération*) | | | |
| B | 24.16 à 24.18 | un réseau PyTorch en estimateur scikit-learn : validation croisée, grille, hasard et zoom (*prochaine session de génération*) | | | |
| C | 24.19 à 24.24 | des CNN sur CIFAR-10 : un 🐛 de traduction, du simple au profond, batchnorm et dropout, augmentation, images synthétiques (*prochaine session de génération*) | | | |
| D | 24.25 à 24.27 | des séries et un LSTM : sommes de sinus, fenêtre, profondeur et données, sorties et état porté sur les taches solaires (*prochaine session de génération*) | | | |
| E | 24.28 à 24.31 | au-delà de nn.Sequential : branches et blocs résiduels, précision mixte et compilation, génération de texte, défi CIFAR-10 frugal (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="24")  # your own library

## Objectifs et rappel express

- Améliorer un réseau par expériences : un hyperparamètre à la fois, dropout et max-norm, un journal de chaque run.
- Régler un réseau PyTorch avec scikit-learn : estimateur maison, `Pipeline`, validation croisée, recherches en grille et au hasard.
- Construire des CNN sur CIFAR-10 (stride, batchnorm, augmentation, blocs résiduels), préparer une série pour un LSTM, lire les sorties de `nn.LSTM`, générer du texte.

**Rappel express.** Mises à jour par epoch : $\lceil N/B \rceil$ ; bruit d'un mini-batch en $1/\sqrt{B}$. Dropout inversé : $\tilde h = m\,h/(1-p)$. Max-norm : $\mathbf{w} \leftarrow \mathbf{w}\min(1, c/\lVert\mathbf{w}\rVert)$. Coût d'une grille : $K \times \prod_j |V_j|$. Convolution : $C_{out}(C_{in}k^2 + 1)$ paramètres, $H_{out}W_{out}C_{out}C_{in}k^2$ MAC. `nn.LSTM(batch_first=True)` : `output` $(N, T, H)$, `h_n` $(L, N, H)$. Bloc résiduel : $y = F(x) + x$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 24.1 à 24.5 et 🧮 24.7)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"F"`) ; plusieurs lettres : entre guillemets (`"AF"`) ; une forme : entre guillemets (`"(3, 5, 1)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ 24.6, la réflexion (🗣️ 24.8, ⚖️ 24.9, 📄 24.10) et l'entretien se corrigent avec `05_solutions.md`.

**24.Q1 — Compter les hyperparamètres et n'en changer qu'un**

In [ ]:
# 24.Q1: the values you found on paper
answer_24_Q1a = ...  # a) the letters, e.g. "AF"
answer_24_Q1b = ...  # b) a whole number
answer_24_Q1c = ...  # c) True or False
answer_24_Q1d = ...  # d) True or False
answer_24_Q1e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_24_Q1a, answer_24_Q1b, answer_24_Q1c, answer_24_Q1d, answer_24_Q1e]):
    wb.check(f"24.Q1{letter}", answer)

**24.Q2 — Taille de batch : bruit, sur-apprentissage et temps**

In [ ]:
# 24.Q2: the values you found on paper
answer_24_Q2a = ...  # a) a whole number
answer_24_Q2b = ...  # b) a whole number
answer_24_Q2c = ...  # c) a number
answer_24_Q2d = ...  # d) True or False
answer_24_Q2e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_24_Q2a, answer_24_Q2b, answer_24_Q2c, answer_24_Q2d, answer_24_Q2e]):
    wb.check(f"24.Q2{letter}", answer)

**24.Q3 — Plus de neurones, moins de neurones, dropout**

In [ ]:
# 24.Q3: the values you found on paper
answer_24_Q3a = ...  # a) the letter of your choice, e.g. "F"
answer_24_Q3b = ...  # b) a number, 1 decimal
answer_24_Q3c = ...  # c) the letter of your choice, e.g. "F"
answer_24_Q3d = ...  # d) a number, 1 decimal
answer_24_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_24_Q3a, answer_24_Q3b, answer_24_Q3c, answer_24_Q3d, answer_24_Q3e]):
    wb.check(f"24.Q3{letter}", answer)

**24.Q4 — Valider un réseau : coût, jeu non représentatif, normalisation dans chaque fold**

In [ ]:
# 24.Q4: the values you found on paper
answer_24_Q4a = ...  # a) a whole number
answer_24_Q4b = ...  # b) the letter of your choice, e.g. "F"
answer_24_Q4c = ...  # c) the letter of your choice, e.g. "F"
answer_24_Q4d = ...  # d) True or False
answer_24_Q4e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_24_Q4a, answer_24_Q4b, answer_24_Q4c, answer_24_Q4d, answer_24_Q4e]):
    wb.check(f"24.Q4{letter}", answer)

**24.Q5 — 🕰️ KerasClassifier, grille, hasard ou zoom : chercher des hyperparamètres**

In [ ]:
# 24.Q5: the values you found on paper
answer_24_Q5a = ...  # a) the letter of your choice, e.g. "F"
answer_24_Q5b = ...  # b) a whole number
answer_24_Q5c = ...  # c) a whole number
answer_24_Q5d = ...  # d) a whole number
answer_24_Q5e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_24_Q5a, answer_24_Q5b, answer_24_Q5c, answer_24_Q5d, answer_24_Q5e]):
    wb.check(f"24.Q5{letter}", answer)

**24.Q6 — 🕰️ Couches utilitaires de Keras et leurs équivalents PyTorch**

In [ ]:
# 24.Q6: the values you found on paper
answer_24_Q6a = ...  # a) the letter of your choice, e.g. "F"
answer_24_Q6b = ...  # b) the letter of your choice, e.g. "F"
answer_24_Q6c = ...  # c) the letter of your choice, e.g. "F"
answer_24_Q6d = ...  # d) the letter of your choice, e.g. "F"
answer_24_Q6e = ...  # e) True or False
answer_24_Q6f = ...  # f) the letter of your choice, e.g. "F"

for letter, answer in zip("abcdef", [answer_24_Q6a, answer_24_Q6b, answer_24_Q6c, answer_24_Q6d, answer_24_Q6e, answer_24_Q6f]):
    wb.check(f"24.Q6{letter}", answer)

**24.Q7 — Préparer des images pour nn.Conv2d**

In [ ]:
# 24.Q7: the values you found on paper
answer_24_Q7a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_24_Q7b = ...  # b) the letter of your choice, e.g. "F"
answer_24_Q7c = ...  # c) a shape as a string, e.g. "(3, 5, 1)"
answer_24_Q7d = ...  # d) True or False
answer_24_Q7e = ...  # e) a number, 1 decimal
answer_24_Q7f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_24_Q7a, answer_24_Q7b, answer_24_Q7c, answer_24_Q7d, answer_24_Q7e, answer_24_Q7f]):
    wb.check(f"24.Q7{letter}", answer)

**24.Q8 — Augmentation et données synthétiques : quelles transformations sont plausibles ?**

In [ ]:
# 24.Q8: the values you found on paper
answer_24_Q8a = ...  # a) the letter of your choice, e.g. "F"
answer_24_Q8b = ...  # b) True or False
answer_24_Q8c = ...  # c) the letter of your choice, e.g. "F"
answer_24_Q8d = ...  # d) the letter of your choice, e.g. "F"
answer_24_Q8e = ...  # e) the letter of your choice, e.g. "F"
answer_24_Q8f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_24_Q8a, answer_24_Q8b, answer_24_Q8c, answer_24_Q8d, answer_24_Q8e, answer_24_Q8f]):
    wb.check(f"24.Q8{letter}", answer)

**24.Q9 — Motifs conv-conv-pool, stride contre pooling, batchnorm contre dropout**

In [ ]:
# 24.Q9: the values you found on paper
answer_24_Q9a = ...  # a) the letter of your choice, e.g. "F"
answer_24_Q9b = ...  # b) True or False
answer_24_Q9c = ...  # c) a number, 2 decimals
answer_24_Q9d = ...  # d) the letter of your choice, e.g. "F"
answer_24_Q9e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_24_Q9a, answer_24_Q9b, answer_24_Q9c, answer_24_Q9d, answer_24_Q9e]):
    wb.check(f"24.Q9{letter}", answer)

**24.Q10 — Séries pour un RNN : fenêtres, scaler, inverse_transform, RMSE**

In [ ]:
# 24.Q10: the values you found on paper
answer_24_Q10a = ...  # a) the letter of your choice, e.g. "F"
answer_24_Q10b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_24_Q10c = ...  # c) the letter of your choice, e.g. "F"
answer_24_Q10d = ...  # d) True or False
answer_24_Q10e = ...  # e) the letter of your choice, e.g. "F"
answer_24_Q10f = ...  # f) the letter of your choice, e.g. "F"

for letter, answer in zip("abcdef", [answer_24_Q10a, answer_24_Q10b, answer_24_Q10c, answer_24_Q10d, answer_24_Q10e, answer_24_Q10f]):
    wb.check(f"24.Q10{letter}", answer)

**24.Q11 — 🕰️ return_sequences, stateful, TimeDistributed : que devient chaque option en PyTorch ?**

In [ ]:
# 24.Q11: the values you found on paper
answer_24_Q11a = ...  # a) the letter of your choice, e.g. "F"
answer_24_Q11b = ...  # b) the letter of your choice, e.g. "F"
answer_24_Q11c = ...  # c) the letter of your choice, e.g. "F"
answer_24_Q11d = ...  # d) the letter of your choice, e.g. "F"
answer_24_Q11e = ...  # e) True or False
answer_24_Q11f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_24_Q11a, answer_24_Q11b, answer_24_Q11c, answer_24_Q11d, answer_24_Q11e, answer_24_Q11f]):
    wb.check(f"24.Q11{letter}", answer)

**24.Q12 — 🕰️ API fonctionnelle de Keras 3 et forward() de PyTorch : branches, partage, gel**

In [ ]:
# 24.Q12: the values you found on paper
answer_24_Q12a = ...  # a) the letter of your choice, e.g. "F"
answer_24_Q12b = ...  # b) the letter of your choice, e.g. "F"
answer_24_Q12c = ...  # c) a shape as a string, e.g. "(3, 5, 1)"
answer_24_Q12d = ...  # d) True or False
answer_24_Q12e = ...  # e) the letter of your choice, e.g. "F"
answer_24_Q12f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_24_Q12a, answer_24_Q12b, answer_24_Q12c, answer_24_Q12d, answer_24_Q12e, answer_24_Q12f]):
    wb.check(f"24.Q12{letter}", answer)

**24.R1 — Ch. 23 : Ce qu'un historique d'entraînement doit contenir pour comparer deux runs**

In [ ]:
# 24.R1: the values you found on paper
answer_24_R1a = ...  # a) the letters, e.g. "AF"
answer_24_R1b = ...  # b) the letters, e.g. "AF"
answer_24_R1c = ...  # c) True or False
answer_24_R1d = ...  # d) a whole number

for letter, answer in zip("abcd", [answer_24_R1a, answer_24_R1b, answer_24_R1c, answer_24_R1d]):
    wb.check(f"24.R1{letter}", answer)

**24.R2 — Ch. 21 : Taille de sortie d'une convolution à padding 'same' et stride 2**

In [ ]:
# 24.R2: the values you found on paper
answer_24_R2a = ...  # a) a whole number
answer_24_R2b = ...  # b) a whole number
answer_24_R2c = ...  # c) a whole number
answer_24_R2d = ...  # d) a whole number
answer_24_R2e = ...  # e) a whole number
answer_24_R2f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_24_R2a, answer_24_R2b, answer_24_R2c, answer_24_R2d, answer_24_R2e, answer_24_R2f]):
    wb.check(f"24.R2{letter}", answer)

**24.R3 — Ch. 17 : Ce que change la division des logits avant le softmax**

In [ ]:
# 24.R3: the values you found on paper
answer_24_R3a = ...  # a) a number, 3 decimals
answer_24_R3b = ...  # b) a number, 3 decimals
answer_24_R3c = ...  # c) a number, 2 decimals
answer_24_R3d = ...  # d) the letter of your choice, e.g. "F"
answer_24_R3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_24_R3a, answer_24_R3b, answer_24_R3c, answer_24_R3d, answer_24_R3e]):
    wb.check(f"24.R3{letter}", answer)

**Ex 24.1 — Le coût d'une grille (27 hyperparamètres, 54 entraînements) contre une recherche aléatoire : probabilité de tomber dans les 5 % meilleurs réglages en n essais**

In [ ]:
# Ex 24.1: the values you found on paper
answer_24_1a = ...  # a) a whole number
answer_24_1b = ...  # b) a number, 1 decimal
answer_24_1c = ...  # c) a whole number
answer_24_1d = ...  # d) a whole number
answer_24_1e = ...  # e) a number, 2 decimals
answer_24_1f = ...  # f) a whole number
answer_24_1g = ...  # g) a number, 3 decimals

for letter, answer in zip("abcdefg", [answer_24_1a, answer_24_1b, answer_24_1c, answer_24_1d, answer_24_1e, answer_24_1f, answer_24_1g]):
    wb.check(f"24.1{letter}", answer)

**Ex 24.2 — Formes et paramètres du CNN à pooling du livre**

In [ ]:
# Ex 24.2: the values you found on paper
answer_24_2a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_24_2b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_24_2c = ...  # c) a whole number
answer_24_2d = ...  # d) a whole number
answer_24_2e = ...  # e) a whole number
answer_24_2f = ...  # f) a whole number
answer_24_2g = ...  # g) a whole number
answer_24_2h = ...  # h) a number, 3 decimals

for letter, answer in zip("abcdefgh", [answer_24_2a, answer_24_2b, answer_24_2c, answer_24_2d, answer_24_2e, answer_24_2f, answer_24_2g, answer_24_2h]):
    wb.check(f"24.2{letter}", answer)

**Ex 24.3 — Stride ou pooling : compter les multiplications**

In [ ]:
# Ex 24.3: the values you found on paper
answer_24_3a = ...  # a) a whole number
answer_24_3b = ...  # b) a whole number
answer_24_3c = ...  # c) a whole number
answer_24_3d = ...  # d) a whole number
answer_24_3e = ...  # e) a number, 2 decimals
answer_24_3f = ...  # f) a whole number
answer_24_3g = ...  # g) a number, 1 decimal

for letter, answer in zip("abcdefg", [answer_24_3a, answer_24_3b, answer_24_3c, answer_24_3d, answer_24_3e, answer_24_3f, answer_24_3g]):
    wb.check(f"24.3{letter}", answer)

**Ex 24.4 — Ce que renvoie nn.LSTM : output, h_n et c_n**

In [ ]:
# Ex 24.4: the values you found on paper
answer_24_4a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_24_4b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_24_4c = ...  # c) the letter of your choice, e.g. "F"
answer_24_4d = ...  # d) a shape as a string, e.g. "(3, 5, 1)"
answer_24_4e = ...  # e) a shape as a string, e.g. "(3, 5, 1)"
answer_24_4f = ...  # f) a whole number
answer_24_4g = ...  # g) a shape as a string, e.g. "(3, 5, 1)"
answer_24_4h = ...  # h) a shape as a string, e.g. "(3, 5, 1)"

for letter, answer in zip("abcdefgh", [answer_24_4a, answer_24_4b, answer_24_4c, answer_24_4d, answer_24_4e, answer_24_4f, answer_24_4g, answer_24_4h]):
    wb.check(f"24.4{letter}", answer)

**Ex 24.5 — Fenêtres, pas d'optimisation et durée : de 200 à 20 000 points**

In [ ]:
# Ex 24.5: the values you found on paper
answer_24_5a = ...  # a) a whole number
answer_24_5b = ...  # b) a whole number
answer_24_5c = ...  # c) a whole number
answer_24_5d = ...  # d) a whole number
answer_24_5e = ...  # e) a number, 1 decimal
answer_24_5f = ...  # f) a whole number
answer_24_5g = ...  # g) a number, 3 decimals

for letter, answer in zip("abcdefg", [answer_24_5a, answer_24_5b, answer_24_5c, answer_24_5d, answer_24_5e, answer_24_5f, answer_24_5g]):
    wb.check(f"24.5{letter}", answer)

**Ex 24.7 — Fermi : le budget d'une recherche d'hyperparamètres sur CPU, sur GPU et en précision mixte**

In [ ]:
# Ex 24.7: the values you found on paper
answer_24_7a = ...  # a) a whole number
answer_24_7b = ...  # b) a whole number
answer_24_7c = ...  # c) a whole number
answer_24_7d = ...  # d) a whole number
answer_24_7e = ...  # e) a number, 2 decimals
answer_24_7f = ...  # f) a number, 1 decimal
answer_24_7g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_24_7a, answer_24_7b, answer_24_7c, answer_24_7d, answer_24_7e, answer_24_7f, answer_24_7g]):
    wb.check(f"24.7{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu mener une amélioration par expériences (un réglage à la fois, un journal) et chiffrer le coût d'une recherche d'hyperparamètres ?
2. Sais-tu envelopper un réseau PyTorch dans un estimateur scikit-learn et le valider sans fuite ?
3. Sais-tu construire un CNN et un LSTM en PyTorch, lire leurs formes, et écrire un bloc résiduel dans `forward()` ?

**Pour aller plus loin** : le « Deep Learning Tuning Playbook » de Google et l'article de He et coll. (2016), cités dans la fiche. La suite : le checkpoint de la partie V, puis le ch. 25, les autoencodeurs.

*Les exercices de code 24.11 à 24.31 (parties A à E : améliorer un MLP, scikit-learn, les CNN sur CIFAR-10, les séries et le LSTM, les architectures à branches, la précision mixte, la génération de texte et le défi) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 24` pour obtenir le notebook complet.*